# VIKOR
---
_**VI**še**K**riterijumska **O**ptimizacija I Kompromisno **R**ešenje_

In [1]:
import numpy as np
import pandas as pd

## 1. Introduction
---
**VIKOR** (_Multicriteria Optimization and Compromise Solution_) ranks alternatives by combining their **total weighted shortfall** from an ideal with their **largest weighted shortfall** on any criterion. It then tests whether the leading alternative is an acceptable **compromise solution**.

This notebook implements the crisp formulation of Opricovic and Tzeng (2004). The compromise can be a single alternative or a set retained for negotiation.

### 1.1. Applications
---
The _VIKOR method_ supports decisions with conflicting criteria where a compromise is acceptable, such as:

- Packaging system selection
- Waste management planning
- Material selection
- Transport technology assessment

Consider **5 packaging systems** for a food service operation. All meet its food contact and minimum containment requirements. Compare **cost** (BRL per delivered meal), **GHG emissions** (kg CO₂e per 1,000 meals), **leakage** (% of deliveries), and **material recovery** (% by mass collected for recycling).

The data and preferences are **illustrative**. Reusable systems include allocated manufacturing, return transport and washing effects over the assumed reuse life. All estimates share the same service and lifecycle boundaries. Cost, GHG and leakage are minimized; recovery is maximized.

In [2]:
df = pd.DataFrame({
    "cost": [0.80, 0.50, 0.55, 1.10, 0.40],
    "GHG": [120, 100, 130, 320, 200],
    "leakage": [1.5, 3.0, 0.8, 0.5, 4.0],
    "recovery": [65, 80, 75, 95, 30]
}, index=[
    "Reusable PP",
    "Molded fiber",
    "Recycled PET",
    "Returnable glass",
    "Coated paper"
])

df.style.background_gradient()

,cost,GHG,leakage,recovery
Reusable PP,0.800000,120,1.500000,65
Molded fiber,0.500000,100,3.000000,80
Recycled PET,0.550000,130,0.800000,75
Returnable glass,1.100000,320,0.500000,95
Coated paper,0.400000,200,4.000000,30


## 2. Method steps
---

### 2.1. Objectives, weights and compromise strategy
---
Define criterion objectives and positive normalized weights. The operator gives leakage the greatest weight, followed by cost and GHG.

The parameter $v\in[0,1]$ controls the balance between the total shortfall and the worst criterion shortfall. Use **$v=0.5$**, giving equal weight to these strategies. This is a decision parameter, rather than a probability.

In [3]:
OBJECTIVES = pd.Series(["min", "min", "min", "max"], index=df.columns)
WEIGHTS = pd.Series([0.25, 0.25, 0.30, 0.20], index=df.columns)

V = 0.5
TOLERANCE = 1e-12

assert (WEIGHTS > 0).all() and np.isclose(WEIGHTS.sum(), 1)
assert 0 <= V <= 1
assert len(df) >= 2

pd.DataFrame({"objective": OBJECTIVES, "weight": WEIGHTS})

,objective,weight
cost,min,0.25
GHG,min,0.25
leakage,min,0.30
recovery,max,0.20


### 2.2. Best and worst performances
---
For criterion $j$, identify its best performance $f_j^*$ and worst performance $f_j^-$ among the alternatives:

$$ \large
(f_j^*,f_j^-)=\begin{cases}
(\max_i x_{ij},\min_i x_{ij}), & \text{if maximized}, \\
(\min_i x_{ij},\max_i x_{ij}), & \text{if minimized}.
\end{cases}
$$

Constant criteria cannot distinguish alternatives. Remove them and renormalize the remaining weights, preserving their relative proportions. All four criteria vary in this example.

In [4]:
BEST = pd.Series(index=df.columns, dtype=float)
WORST = pd.Series(index=df.columns, dtype=float)

for col in df.columns:
    if OBJECTIVES[col] == "max":
        BEST[col], WORST[col] = df[col].max(), df[col].min()
    else:
        BEST[col], WORST[col] = df[col].min(), df[col].max()

active = BEST != WORST
CRITERIA = df.columns[active]

if len(CRITERIA) == 0:
    raise ValueError("All criteria are constant; the alternatives cannot be distinguished.")

WEIGHTS = WEIGHTS.loc[CRITERIA]
WEIGHTS = WEIGHTS/WEIGHTS.sum()

pd.DataFrame({"best": BEST, "worst": WORST, "active": active})

,best,worst,active
cost,0.4,1.1,True
GHG,100.0,320.0,True
leakage,0.5,4.0,True
recovery,95.0,30.0,True


### 2.3. Normalized shortfalls
---
Normalize the shortfall from the ideal:

$$ \large
d_{ij}=\frac{f_j^*-x_{ij}}{f_j^*-f_j^-}
$$

This expression handles both objectives: the numerator and denominator are both negative for a cost criterion. Values range from **0** (best) to **1** (worst). The reference performances depend on the alternatives being compared.

In [5]:
df_n = (BEST[CRITERIA] - df[CRITERIA])/(BEST[CRITERIA] - WORST[CRITERIA])

df_n.style.background_gradient(vmin=0, vmax=1)

,cost,GHG,leakage,recovery
Reusable PP,0.571429,0.090909,0.285714,0.461538
Molded fiber,0.142857,-0.000000,0.714286,0.230769
Recycled PET,0.214286,0.136364,0.085714,0.307692
Returnable glass,1.000000,1.000000,-0.000000,0.000000
Coated paper,-0.000000,0.454545,1.000000,1.000000


### 2.4. Weighted shortfalls
---
Multiply each normalized shortfall by its criterion weight:

$$ \large
e_{ij}=w_j d_{ij}
$$

The following table shows the contribution of every criterion to each alternative's shortfall.

In [6]:
df_w = df_n*WEIGHTS

df_w

,cost,GHG,leakage,recovery
Reusable PP,0.142857,0.022727,0.085714,0.092308
Molded fiber,0.035714,-0.000000,0.214286,0.046154
Recycled PET,0.053571,0.034091,0.025714,0.061538
Returnable glass,0.250000,0.250000,-0.000000,0.000000
Coated paper,-0.000000,0.113636,0.300000,0.200000


### 2.5. Group utility and individual regret measures
---
Calculate the **total shortfall** $S_i$ and the **largest individual shortfall** $R_i$:

$$ \large
S_i=\sum_j e_{ij}, \qquad R_i=\max_j e_{ij}
$$

Both are minimized. Lower $S_i$ represents greater overall group utility; lower $R_i$ limits the worst weighted criterion regret. Neither quantity is a probability or monetary loss.

In [7]:
S = df_w.sum(axis=1)
R = df_w.max(axis=1)

pd.DataFrame({"S": S, "R": R, "largest shortfall criterion": df_w.idxmax(axis=1)})

,S,R,largest shortfall criterion
Reusable PP,0.343606,0.142857,cost
Molded fiber,0.296154,0.214286,leakage
Recycled PET,0.174915,0.061538,recovery
Returnable glass,0.500000,0.250000,cost
Coated paper,0.613636,0.300000,leakage


### 2.6. Compromise index
---
Combine the two measures:

$$ \large
Q_i=v\frac{S_i-S^*}{S^--S^*}+(1-v)\frac{R_i-R^*}{R^--R^*}
$$

Here $S^*=\min_iS_i$, $S^-=\max_iS_i$, $R^*=\min_iR_i$ and $R^-=\max_iR_i$. Lower $Q_i$ is preferred.

The standard expression is undefined if either denominator is zero. This recipe stops explicitly in that case, so a tied measure is reviewed instead of introducing an arbitrary numerical offset.

In [8]:
S_best, S_worst = S.min(), S.max()
R_best, R_worst = R.min(), R.max()

if (
    np.isclose(S_best, S_worst, atol=TOLERANCE, rtol=0)
    or np.isclose(R_best, R_worst, atol=TOLERANCE, rtol=0)
):
    raise ValueError("The standard Q expression is undefined for a constant S or R measure.")

S_n = (S - S_best)/(S_worst - S_best)
R_n = (R - R_best)/(R_worst - R_best)
Q = V*S_n + (1 - V)*R_n

pd.DataFrame({"normalized S": S_n, "normalized R": R_n, "Q": Q})

,normalized S,normalized R,Q
Reusable PP,0.384507,0.341014,0.362760
Molded fiber,0.276346,0.640553,0.458449
Recycled PET,0.000000,0.000000,0.000000
Returnable glass,0.740983,0.790323,0.765653
Coated paper,1.000000,1.000000,1.000000


### 2.7. Three ranking lists
---
Rank alternatives in **ascending order** of $S$, $R$ and $Q$. Equal values share a rank. The $Q$ ordering is the compromise ranking list; recommending its leader requires the following acceptance tests.

In [9]:
df_metrics = pd.DataFrame({"S": S, "R": R, "Q": Q})

for col in ["S", "R", "Q"]:
    df_metrics[f'rank {col}'] = df_metrics[col].rank(method="min").astype(int)

df_order = df_metrics.sort_values("Q", kind="stable")

df_order

,S,R,Q,rank S,rank R,rank Q
Recycled PET,0.174915,0.061538,0.000000,1,1,1
Reusable PP,0.343606,0.142857,0.362760,3,2,2
Molded fiber,0.296154,0.214286,0.458449,2,3,3
Returnable glass,0.500000,0.250000,0.765653,4,4,4
Coated paper,0.613636,0.300000,1.000000,5,5,5


### 2.8. Acceptable advantage and stability
---
Let $a^{(1)}$ and $a^{(2)}$ lead the $Q$ list, with $m$ alternatives. Check:

$$ \large
C1:\quad Q(a^{(2)})-Q(a^{(1)})\geq DQ, \qquad DQ=\frac{1}{m-1}
$$

**C2** requires $a^{(1)}$ to also be best by **$S$ or $R$**, allowing tied best values. This is stability in the acceptance rule, not a general sensitivity guarantee.

In [10]:
first, second = df_order.index[:2]
DQ = 1/(len(df) - 1)
Q_gap = Q[second] - Q[first]

acceptable_advantage = Q_gap >= DQ - TOLERANCE
acceptable_stability = (
    np.isclose(S[first], S_best, atol=TOLERANCE, rtol=0)
    or np.isclose(R[first], R_best, atol=TOLERANCE, rtol=0)
)

pd.Series({
    "Q gap": Q_gap,
    "DQ": DQ,
    "C1: acceptable advantage": acceptable_advantage,
    "C2: acceptable stability": acceptable_stability
})

Q gap                       0.36276
DQ                             0.25
C1: acceptable advantage       True
C2: acceptable stability       True
dtype: object

### 2.9. Compromise recommendation
---
Apply the acceptance rule:

- If **both conditions hold**, retain $a^{(1)}$ alone.
- If **C1 fails**, retain all alternatives with $Q_i-Q(a^{(1)})<DQ$, even if C2 also fails.
- If **only C2 fails**, retain the first **two** alternatives.

The second rule describes a close group for further discussion, rather than an arbitrary tie-breaking rule.

In [11]:
if not acceptable_advantage:
    COMPROMISE = list(df_order.index[(df_order["Q"] - Q[first]) < DQ - TOLERANCE])
elif not acceptable_stability:
    COMPROMISE = list(df_order.index[:2])
else:
    COMPROMISE = [first]

df_result = df.join(df_metrics)
df_result["compromise"] = df_result.index.isin(COMPROMISE)

df_result.sort_values("Q").style.background_gradient(subset=list(CRITERIA))

,cost,GHG,leakage,recovery,S,R,Q,rank S,rank R,rank Q,compromise
Recycled PET,0.550000,130,0.800000,75,0.174915,0.061538,0.000000,1,1,1,True
Reusable PP,0.800000,120,1.500000,65,0.343606,0.142857,0.362760,3,2,2,False
Molded fiber,0.500000,100,3.000000,80,0.296154,0.214286,0.458449,2,3,3,False
Returnable glass,1.100000,320,0.500000,95,0.500000,0.250000,0.765653,4,4,4,False
Coated paper,0.400000,200,4.000000,30,0.613636,0.300000,1.000000,5,5,5,False


**Recycled PET** is the unique accepted compromise. It is not best on any individual criterion, but it has both the lowest total shortfall and the lowest worst weighted shortfall. The Q gap to Reusable PP exceeds the required advantage.

Its **Q = 0** means that it is best on both aggregate measures among the available systems; it does not mean that it achieves every criterion's ideal performance. Glass has less leakage and greater recovery, while molded fiber has lower GHG, but their other disadvantages increase their compromise measures.

### 2.10. Strategy sensitivity
---
Repeat the compromise checks for nearby values of $v$. This checks sensitivity to the balance between total shortfall and the worst shortfall, with the data and criterion weights fixed.

In [12]:
sensitivity = []

for v in [0.25, 0.50, 0.75]:
    q = v*S_n + (1 - v)*R_n
    order = q.sort_values(kind="stable")
    leader = order.index[0]
    advantage = order.iloc[1] - order.iloc[0] >= DQ - TOLERANCE
    stability = (
        np.isclose(S[leader], S_best, atol=TOLERANCE, rtol=0)
        or np.isclose(R[leader], R_best, atol=TOLERANCE, rtol=0)
    )

    if not advantage:
        compromise = list(order.index[(order - order.iloc[0]) < DQ - TOLERANCE])
    elif not stability:
        compromise = list(order.index[:2])
    else:
        compromise = [leader]

    sensitivity.append([v, leader, advantage, stability, compromise])

pd.DataFrame(sensitivity, columns=["v", "Q leader", "C1", "C2", "compromise"])

,v,Q leader,C1,C2,compromise
0,0.25,Recycled PET,True,True,[Recycled PET]
1,0.50,Recycled PET,True,True,[Recycled PET]
2,0.75,Recycled PET,True,True,[Recycled PET]


## 3. Examples
---

### 3.1. Municipal waste management
---
A municipality compares **4 waste management plans** for the same annual waste stream. Evaluate **annual system cost** (BRL million), **landfill diversion** (% by mass), **annual system GHG** (thousand tonnes CO₂e), and **implementation time** (months). Cost, GHG and time are minimized; diversion is maximized.

The data and preferences are **illustrative**, with common waste quantities and emissions boundaries. Use **v = 0.5** and the weights below. Unlike the packaging example, this case deliberately illustrates a **close compromise set**, rather than a unique recommendation.

In [13]:
df_ex1 = pd.DataFrame({
    "cost": [6.5, 6.0, 5.0, 10.0],
    "diversion": [65, 59, 20, 80],
    "GHG": [3.6, 4.4, 8.4, 2.0],
    "implementation": [10, 8, 4, 24]
}, index=[
    "Organics and collection",
    "Sorting and education",
    "Awareness campaign",
    "Integrated treatment"
])

df_ex1.style.background_gradient()

,cost,diversion,GHG,implementation
Organics and collection,6.500000,65,3.600000,10
Sorting and education,6.000000,59,4.400000,8
Awareness campaign,5.000000,20,8.400000,4
Integrated treatment,10.000000,80,2.000000,24


In [14]:
OBJECTIVES_ex1 = pd.Series(["min", "max", "min", "min"], index=df_ex1.columns)
WEIGHTS_ex1 = pd.Series([0.30, 0.30, 0.25, 0.15], index=df_ex1.columns)
V_ex1 = 0.5

pd.DataFrame({"objective": OBJECTIVES_ex1, "weight": WEIGHTS_ex1})

,objective,weight
cost,min,0.30
diversion,max,0.30
GHG,min,0.25
implementation,min,0.15


In [15]:
# Best and worst performances
BEST_ex1 = pd.Series(index=df_ex1.columns, dtype=float)
WORST_ex1 = pd.Series(index=df_ex1.columns, dtype=float)

for col in df_ex1.columns:
    if OBJECTIVES_ex1[col] == "max":
        BEST_ex1[col], WORST_ex1[col] = df_ex1[col].max(), df_ex1[col].min()
    else:
        BEST_ex1[col], WORST_ex1[col] = df_ex1[col].min(), df_ex1[col].max()

assert (BEST_ex1 != WORST_ex1).all()

pd.DataFrame({"best": BEST_ex1, "worst": WORST_ex1})

,best,worst
cost,5.0,10.0
diversion,80.0,20.0
GHG,2.0,8.4
implementation,4.0,24.0


In [16]:
# Normalized shortfalls
df_ex1_n = (BEST_ex1 - df_ex1)/(BEST_ex1 - WORST_ex1)

df_ex1_n

,cost,diversion,GHG,implementation
Organics and collection,0.3,0.25,0.250,0.3
Sorting and education,0.2,0.35,0.375,0.2
Awareness campaign,-0.0,1.00,1.000,-0.0
Integrated treatment,1.0,0.00,-0.000,1.0


In [17]:
# Weighted contributions
df_ex1_w = df_ex1_n*WEIGHTS_ex1

df_ex1_w

,cost,diversion,GHG,implementation
Organics and collection,0.09,0.075,0.06250,0.045
Sorting and education,0.06,0.105,0.09375,0.030
Awareness campaign,-0.00,0.300,0.25000,-0.000
Integrated treatment,0.30,0.000,-0.00000,0.150


In [18]:
# Group utility, regret and compromise index
S_ex1 = df_ex1_w.sum(axis=1)
R_ex1 = df_ex1_w.max(axis=1)

if (
    np.isclose(S_ex1.min(), S_ex1.max(), atol=TOLERANCE, rtol=0)
    or np.isclose(R_ex1.min(), R_ex1.max(), atol=TOLERANCE, rtol=0)
):
    raise ValueError("The standard Q expression is undefined for a constant S or R measure.")

Q_ex1 = (
    V_ex1*(S_ex1 - S_ex1.min())/(S_ex1.max() - S_ex1.min())
    + (1 - V_ex1)*(R_ex1 - R_ex1.min())/(R_ex1.max() - R_ex1.min())
)

df_ex1_metrics = pd.DataFrame({"S": S_ex1, "R": R_ex1, "Q": Q_ex1})

for col in ["S", "R", "Q"]:
    df_ex1_metrics[f'rank {col}'] = df_ex1_metrics[col].rank(method="min").astype(int)

df_ex1_order = df_ex1_metrics.sort_values("Q", kind="stable")

df_ex1_order

,S,R,Q,rank S,rank R,rank Q
Organics and collection,0.27250,0.090,0.000000,1,1,1
Sorting and education,0.28875,0.105,0.064994,2,2,2
Integrated treatment,0.45000,0.300,0.819820,3,3,3
Awareness campaign,0.55000,0.300,1.000000,4,3,4


In [19]:
# Compromise acceptance checks
first_ex1, second_ex1 = df_ex1_order.index[:2]
DQ_ex1 = 1/(len(df_ex1) - 1)
Q_gap_ex1 = Q_ex1[second_ex1] - Q_ex1[first_ex1]

advantage_ex1 = Q_gap_ex1 >= DQ_ex1 - TOLERANCE
stability_ex1 = (
    np.isclose(S_ex1[first_ex1], S_ex1.min(), atol=TOLERANCE, rtol=0)
    or np.isclose(R_ex1[first_ex1], R_ex1.min(), atol=TOLERANCE, rtol=0)
)

pd.Series({
    "Q gap": Q_gap_ex1,
    "DQ": DQ_ex1,
    "C1: acceptable advantage": advantage_ex1,
    "C2: acceptable stability": stability_ex1
})

Q gap                       0.064994
DQ                          0.333333
C1: acceptable advantage       False
C2: acceptable stability        True
dtype: object

In [20]:
if not advantage_ex1:
    COMPROMISE_ex1 = list(
        df_ex1_order.index[(df_ex1_order["Q"] - Q_ex1[first_ex1]) < DQ_ex1 - TOLERANCE]
    )
elif not stability_ex1:
    COMPROMISE_ex1 = list(df_ex1_order.index[:2])
else:
    COMPROMISE_ex1 = [first_ex1]

df_ex1_result = df_ex1.join(df_ex1_metrics)
df_ex1_result["compromise"] = df_ex1_result.index.isin(COMPROMISE_ex1)

df_ex1_result.sort_values("Q").style.background_gradient(subset=list(df_ex1.columns))

,cost,diversion,GHG,implementation,S,R,Q,rank S,rank R,rank Q,compromise
Organics and collection,6.500000,65,3.600000,10,0.272500,0.090000,0.000000,1,1,1,True
Sorting and education,6.000000,59,4.400000,8,0.288750,0.105000,0.064994,2,2,2,True
Integrated treatment,10.000000,80,2.000000,24,0.450000,0.300000,0.819820,3,3,3,False
Awareness campaign,5.000000,20,8.400000,4,0.550000,0.300000,1.000000,4,3,4,False


**Organics and collection** leads the Q list and is also best by S and R. However, its Q advantage over **Sorting and education** is below **DQ = 1/3**. The accepted compromise set contains **both plans**; reporting only the Q leader would omit the advantage test.

Organics provides stronger diversion and lower emissions, while sorting costs less and can be implemented sooner. Integrated treatment has the best environmental performances but its cost and implementation time create large shortfalls. The set gives the municipality two close options for negotiation under its stated preferences.

## 4. References
---

- Opricovic, S. and Tzeng, G.-H. (2004). [*Compromise solution by MCDM methods: A comparative analysis of VIKOR and TOPSIS*](https://doi.org/10.1016/S0377-2217(03)00020-1). Defines the S, R and Q measures and the compromise acceptance conditions.
- Opricovic, S. and Tzeng, G.-H. (2007). [*Extended VIKOR method in comparison with outranking methods*](https://doi.org/10.1016/j.ejor.2006.01.020).